<a href="https://colab.research.google.com/github/AuliaRestyy/machineLearning/blob/js09/JS09_Assignment1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install numpy annoy faiss-cpu hnswlib

In [8]:
import numpy as np
import time
import faiss
from annoy import AnnoyIndex
import hnswlib
import pandas as pd

In [9]:
# Dataset configurations

datasets = [
    (1000, 2),
    (1000, 5),
    (1000000, 2),
    (1000000, 5)
]

k = 10
metrics = ['euclidean', 'cosine']

In [11]:
# Experiment

for n_data, dim in datasets:

    print("\n=====================================")
    print("DATASET:", n_data, "data,", dim, "dimensions")
    print("=====================================")

    X = np.random.random((n_data, dim)).astype(np.float32)
    query = np.random.random((1, dim)).astype(np.float32)

    for metric in metrics:

        print("\n----------", metric.upper(), "----------")

        X_test = X.copy()
        query_test = query.copy()

        # 1. ANNOY

        print("\n=== ANNOY ===")

        if metric == 'euclidean':
            ann_index = AnnoyIndex(dim, 'euclidean')
        else:
            ann_index = AnnoyIndex(dim, 'angular')

        start = time.time()

        for i in range(n_data):
            ann_index.add_item(i, X_test[i])

        ann_index.build(10)

        build_time = time.time() - start

        start = time.time()

        neighbors = ann_index.get_nns_by_vector(
            query_test[0], k, include_distances=True
        )

        query_time = time.time() - start

        print("Build time:", build_time, "seconds")
        print("Query time:", query_time, "seconds")
        print("Neighbors:", neighbors[0][:5])

        # 2. FAISS

        print("\n=== FAISS ===")

        if metric == 'euclidean':

            quantizer = faiss.IndexFlatL2(dim)

            faiss_index = faiss.IndexIVFFlat(
                quantizer, dim, 10, faiss.METRIC_L2
            )

        else:

            faiss.normalize_L2(X_test)
            faiss.normalize_L2(query_test)

            quantizer = faiss.IndexFlatIP(dim)

            faiss_index = faiss.IndexIVFFlat(
                quantizer, dim, 10, faiss.METRIC_INNER_PRODUCT
            )

        start = time.time()

        faiss_index.train(X_test)
        faiss_index.add(X_test)

        faiss_index.nprobe = 5

        build_time = time.time() - start

        start = time.time()

        distances, indices = faiss_index.search(query_test, k)

        query_time = time.time() - start

        print("Build time:", build_time, "seconds")
        print("Query time:", query_time, "seconds")
        print("Neighbors:", indices[0][:5])

        # 3. HNSW

        print("\n=== HNSW ===")

        if metric == 'euclidean':
            hnsw_index = hnswlib.Index(space='l2', dim=dim)
        else:
            hnsw_index = hnswlib.Index(space='cosine', dim=dim)

        start = time.time()

        hnsw_index.init_index(
            max_elements=n_data,
            ef_construction=200,
            M=16
        )

        hnsw_index.add_items(X, np.arange(n_data))

        hnsw_index.set_ef(50)

        build_time = time.time() - start

        start = time.time()

        labels, distances = hnsw_index.knn_query(query, k=k)

        query_time = time.time() - start

        print("Build time:", build_time, "seconds")
        print("Query time:", query_time, "seconds")
        print("Neighbors:", labels[0][:5])

        del ann_index
        del faiss_index
        del hnsw_index

    del X



DATASET: 1000 data, 2 dimensions

---------- EUCLIDEAN ----------

=== ANNOY ===
Build time: 0.056464433670043945 seconds
Query time: 7.557868957519531e-05 seconds
Neighbors: [125, 992, 905, 832, 246]

=== FAISS ===
Build time: 0.1514437198638916 seconds
Query time: 8.416175842285156e-05 seconds
Neighbors: [125 992 905 832 246]

=== HNSW ===
Build time: 0.1116182804107666 seconds
Query time: 9.202957153320312e-05 seconds
Neighbors: [125 992 905 832 246]

---------- COSINE ----------

=== ANNOY ===
Build time: 0.061403751373291016 seconds
Query time: 7.581710815429688e-05 seconds
Neighbors: [296, 52, 90, 0, 521]

=== FAISS ===
Build time: 0.14792132377624512 seconds
Query time: 7.700920104980469e-05 seconds
Neighbors: [296  52  90   0 521]

=== HNSW ===
Build time: 0.09752869606018066 seconds
Query time: 6.532669067382812e-05 seconds
Neighbors: [296  52  90   0 521]

DATASET: 1000 data, 5 dimensions

---------- EUCLIDEAN ----------

=== ANNOY ===
Build time: 0.029163599014282227 second